In [1]:
# Enable auto-reload for imported modules
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Get project root (from training/notebooks/ go up 2 levels)
project_root = Path.cwd().parent.parent  

# Add paths
sys.path.insert(0, str(project_root))

# Verify paths
print("✓ Paths added to sys.path")

from training import settings
import os

os.environ["TRANSFORMERS_CACHE"] = str(settings.TRANSFORMER_CACHE_DIR)
os.environ["HF_HOME"] = str(settings.TRANSFORMER_DATASETS_DIR)

✓ Paths added to sys.path


In [ ]:
from src.payment_classifier.inference.prob_inference import ProbModelInference
from app.core.schemas import PAYMENT_LABEL_V2

inferencer = ProbModelInference("/Users/maroon/workspace/tiny-model-tunning/finetune/.checkpoints/payment_classification_v2/8", label_config=PAYMENT_LABEL_V2)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
inputs = [
    # "send 4$"
    # "pay 5$ now"
    # "hahaha"
    # "omg"
    # "Hi there"
    # "This is for a test payment.",
    # "Can you send me a payment request?",
    # "What is the weather like today?"
    # "Hi",
    # "Hello",
    # "Hey guys",
    # "Good morning",
    # "I would like to schedule a meeting.",
    # "Give me back 3$ you own"
    # "Give me back my 3$"
    # "Give him 3"
    # "Please send me the invoice.",
    # "Can you help me with my account?",
    # "What is the status of my order?",
    # "I need assistance with my payment.",
    # "Thank you for your help.",
    # "Same arrangement as before?",
    # "I'll clean the dinner"
]
predict = inferencer.predict(inputs)
import json
print(predict)

[{'label': 'open_intent', 'prob': 0.47282007336616516, 'all_probs': {'payment_request': 0.32794591784477234, 'payment_intent': 0.1992340087890625, 'open_intent': 0.47282007336616516}}]


In [2]:
from app.core import schemas
from src.payment_classifier.llm.base import BaseLLM
from src.payment_classifier.prompts.base import BasePromptManager
import typing as t
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel

def classify_errors_by_clustering(errors: t.List[schemas.Sample]):
    # Initialize embedding model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Step 2: Generate embeddings for all errors
    error_texts = [error.msg for error in errors]
    embeddings = model.encode(error_texts)
    
    # Step 3: Apply clustering
    from sklearn.cluster import DBSCAN
    clustering = DBSCAN(eps=0.5, min_samples=2, metric='cosine')
    cluster_labels = clustering.fit_predict(embeddings)
    
    # Step 4: Group errors by cluster
    buckets = {}
    for idx, label in enumerate(cluster_labels):
        if label not in buckets:
            buckets[label] = []
        buckets[label].append(errors[idx])
    
    return buckets

/Users/maroon/.pyenv/versions/3.12.7/envs/tiny-model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/maroon/.pyenv/versions/3.12.7/envs/tiny-model/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [12]:
output = classify_errors_by_clustering([
schemas.Sample(msg="Can u plz send me fifty bucks when ur ready?", label="payment_request"),
schemas.Sample(msg="I'll zap you back the $100 tmrw via Venmo", label="payment_intent"),
schemas.Sample(msg="Don't forget to transfer me and John €50 each", label="payment_request"),
schemas.Sample(msg="Should I send him 1000 sats or pay attention to the meeting?", label="other"),
schemas.Sample(msg="Hook me up with that fifty dollars u owe from last week", label="payment_request"),
schemas.Sample(msg="I'm gonna chip in for the escrow payment through my PayPal account", label="payment_intent"),
schemas.Sample(msg="Won't send you the 0.5 BTC unless everyone splits it three ways", label="payment_intent"),
schemas.Sample(msg="plsss PAY ME BACK!!!! that donation pledge money", label="payment_request"),
schemas.Sample(msg="Can u toss me and Sarah 1k USD each to our Lightning invoices?", label="payment_request"),
schemas.Sample(msg="Should I charge my phone or charge you the subscription refund?", label="other"),
])

In [15]:

total = sum(len(v) for v in output.values())

for v in output.values():
    print(list(v))

[Sample(msg='Can u plz send me fifty bucks when ur ready?', label='payment_request'), Sample(msg='Hook me up with that fifty dollars u owe from last week', label='payment_request')]
[Sample(msg="I'll zap you back the $100 tmrw via Venmo", label='payment_intent'), Sample(msg="Don't forget to transfer me and John €50 each", label='payment_request'), Sample(msg='Should I send him 1000 sats or pay attention to the meeting?', label='other'), Sample(msg="I'm gonna chip in for the escrow payment through my PayPal account", label='payment_intent'), Sample(msg="Won't send you the 0.5 BTC unless everyone splits it three ways", label='payment_intent'), Sample(msg='plsss PAY ME BACK!!!! that donation pledge money', label='payment_request'), Sample(msg='Can u toss me and Sarah 1k USD each to our Lightning invoices?', label='payment_request'), Sample(msg='Should I charge my phone or charge you the subscription refund?', label='other')]


In [ ]:
from app.core.services import ErrorCategorizer
from src.payment_classifier.llm.litellm import LiteLLMProvider
from src.payment_classifier.llm.settings import LLMSettings
from src.payment_classifier.prompts import InmemoryPromptManager
from app.core.settings import settings
from app.core.schemas import PAYMENT_LABEL_V2

p = ErrorCategorizer(
    llm=LiteLLMProvider(
        LLMSettings(
            api_key=settings.OPENAI_API_KEY,
            model_name="gpt-4.1",
            temperature=0.2,
        )
    ),
    prompt_mgr=InmemoryPromptManager(),
    label_config=PAYMENT_LABEL_V2,
)



In [ ]:
from app.core import schemas

prompt = ""

with open("/Users/maroon/workspace/tiny-model-tunning/finetune/app/core/prompts/v2/analyze/categorize_error.txt", "r") as f:
    prompt = f.read()


output = await p.categorize_testcase(
    schemas.TestCase(
        input=schemas.Sample(msg="Can u plz send me the doc when ur ready?", label="payment_request"),
        true_label="open_intent",
        prediction=schemas.Prediction(
            label="payment_request",
            prob=0.4,
        )
    ),
    prompt
)


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/pro

InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers LiteLLM Retried: 5 times
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers LiteLLM Retried: 5 times
</exception>
<completion>
    None
</completion>
</generation>

<generation number="3">
<exception>
    litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying